In [1]:
import pandas as pd
import numpy as np

from tqdm import tqdm

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


# =========================================================
# LOAD DATA
# =========================================================

df = pd.read_csv(
    "career_role_prediction.csv"
)

print("Data:", len(df))


# =========================================================
# CLEAN TEXT
# =========================================================

df["title"] = (
    df["title"]
    .fillna("")
    .astype(str)
    .str.lower()
)

df["body"] = (
    df["body"]
    .fillna("")
    .astype(str)
    .str.lower()
)

df["text"] = (
    df["title"] + " " +
    df["body"]
)


# =========================================================
# LOAD MODEL
# =========================================================

print("\nLoading model...")

model = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2"
)


# =========================================================
# REFERENCE EXAMPLES
# =========================================================

problem_examples = {

    "confidence_issue": [

        "i am not confident",
        "i doubt myself",
        "i feel insecure",
        "i compare myself to others",
        "i feel stupid",
        "i am scared to apply",
        "i have imposter syndrome"

    ],

    "overwhelmed": [

        "there is too much to learn",
        "i feel overwhelmed",
        "too many things to study",
        "i dont know what to learn first",
        "everything feels confusing",
        "my roadmap is messy"

    ],

    "direction_confused": [

        "i dont know which path to choose",
        "frontend or backend",
        "which tech career is best",
        "i keep changing career path",
        "i am confused about my future"

    ],

    "beginner_lost": [

        "where do i start",
        "i am a complete beginner",
        "i have zero experience",
        "i dont know how to start coding",
        "i am new to tech"

    ]

}


# =========================================================
# LEVEL EXAMPLES
# =========================================================

level_examples = {

    "zero": [

        "i have zero experience",
        "i never coded before",
        "complete beginner",
        "starting from zero"

    ],

    "basic": [

        "i finished tutorials",
        "i know basic python",
        "i made small projects",
        "i joined bootcamp"

    ],

    "intermediate": [

        "i have internship experience",
        "i worked on projects",
        "i am intermediate",
        "i freelance sometimes"

    ]

}


# =========================================================
# BLOCKER EXAMPLES
# =========================================================

blocker_examples = {

    "too_many_options": [

        "too many choices",
        "too many roadmaps",
        "i keep changing path",
        "i cannot decide",
        "i jump between topics"

    ],

    "no_portfolio": [

        "i have no projects",
        "my portfolio is empty",
        "nothing to show employers",
        "i lack practical projects"

    ],

    "no_time": [

        "i work full time",
        "i am busy every day",
        "i dont have time to study",
        "my schedule is packed"

    ],

    "no_confidence": [

        "i doubt myself",
        "i feel insecure",
        "i compare myself to others",
        "i am afraid to apply"

    ],

    "no_foundation": [

        "my fundamentals are weak",
        "i dont understand basics",
        "my coding basics are bad",
        "foundation is weak"

    ]

}


# =========================================================
# GENERIC FUNCTION
# =========================================================

def predict_label(text, label_examples):

    # =====================================================
    # ENCODE INPUT
    # =====================================================

    text_emb = model.encode(
        [text],
        normalize_embeddings=True
    )


    best_label = None
    best_score = -1


    # =====================================================
    # LOOP LABELS
    # =====================================================

    for label, examples in label_examples.items():

        example_embs = model.encode(
            examples,
            normalize_embeddings=True
        )

        sims = cosine_similarity(
            text_emb,
            example_embs
        )[0]

        score = np.mean(
            np.sort(sims)[-2:]
        )

        if score > best_score:

            best_score = score
            best_label = label


    return best_label, round(float(best_score), 4)


# =========================================================
# PREDICTION
# =========================================================

problem_labels = []
level_labels = []
blocker_labels = []

print("\nPredicting labels...")

for text in tqdm(df["text"]):

    # =====================================================
    # PROBLEM
    # =====================================================

    problem, _ = predict_label(
        text,
        problem_examples
    )


    # =====================================================
    # LEVEL
    # =====================================================

    level, _ = predict_label(
        text,
        level_examples
    )


    # =====================================================
    # BLOCKER
    # =====================================================

    blocker, _ = predict_label(
        text,
        blocker_examples
    )


    # =====================================================
    # SAVE
    # =====================================================

    problem_labels.append(problem)

    level_labels.append(level)

    blocker_labels.append(blocker)


# =========================================================
# SAVE RESULT
# =========================================================

df["problem_category"] = problem_labels

df["current_level"] = level_labels

df["blocker_type"] = blocker_labels


# =========================================================
# PREVIEW
# =========================================================

preview_cols = [

    "title",
    "predicted_role",
    "problem_category",
    "current_level",
    "blocker_type"

]

print("\nTOP RESULTS:\n")

print(
    df[preview_cols]
    .head(20)
)


# =========================================================
# SAVE CSV
# =========================================================

output_path = "career_final_dataset.csv"

df.to_csv(

    output_path,

    index=False,
    encoding="utf-8-sig"

)

print("\nDONE!")
print("Saved to:", output_path)


C:\Users\Gregorius Christian\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data: 2942

Loading model...

Predicting labels...


 54%|█████▎    | 1576/2942 [43:24<37:37,  1.65s/it]  


RuntimeError: [enforce fail at alloc_cpu.cpp:117] data. DefaultCPUAllocator: not enough memory: you tried to allocate 6255408 bytes.

In [ ]:
df = pd.read_csv("career_labeled_dataset.csv")

In [ ]:
df.head()

,title,body,dominant_topic_name,url,good_score,bad_score,keyword_match,negative_match,final_score,career_text,predicted_role,role_score,top3_roles,top3_scores,text,problem_category,current_level,blocker_type
0,what are the skills required for vfx artist?,passionate about vfx. wanted to know more abou...,Career Change for Non-Corporate Artist,https://www.reddit.com/r/careerguidance/commen...,0.3588,0.1249,False,False,0.3307,what are the skills required for vfx artist? p...,vfx artist,0.7777,"['vfx artist', 'visual effects animator', 'gra...","[0.7777, 0.6448, 0.6029]",what are the skills required for vfx artist? p...,unclear,unclear,unclear
1,web developers out there can you provide guida...,i am a 4th year b.tech student (electronics an...,Software Development Career Transition,https://www.reddit.com/r/careerguidance/commen...,0.5111,0.2009,True,False,0.5326,web developers out there can you provide guida...,entry level web developer,0.7579,"['entry level web developer', 'web developer',...","[0.7579, 0.7473, 0.7355]",web developers out there can you provide guida...,unclear,unclear,unclear
2,interested in developing a career in the busin...,hello! i m a business administration and i m r...,Career Advancement in Data Analytics,https://www.reddit.com/r/careerguidance/commen...,0.5177,0.3283,True,False,0.4385,interested in developing a career in the busin...,business intelligence analyst,0.7509,"['business intelligence analyst', 'power bi an...","[0.7509, 0.6451, 0.589]",interested in developing a career in the busin...,overwhelmed,unclear,unclear
3,what are possible career s paths for data busi...,hi i am working in a company as data analyst d...,Career Advancement in Data Analytics,https://www.reddit.com/r/careerguidance/commen...,0.6199,0.3609,True,False,0.5352,what are possible career s paths for data busi...,business intelligence analyst,0.7503,"['business intelligence analyst', 'power bi an...","[0.7503, 0.6684, 0.6515]",what are possible career s paths for data busi...,direction_confused,intermediate,unclear
4,what are the best opportunities for a manageme...,hi r careerguidance! i am currently a manageme...,Career Advancement in Data Analytics,http://www.reddit.com/r/careerguidance/comment...,0.4888,0.3160,True,False,0.4138,what are the best opportunities for a manageme...,business intelligence analyst,0.7425,"['business intelligence analyst', 'power bi an...","[0.7425, 0.6556, 0.5415]",what are the best opportunities for a manageme...,unclear,unclear,unclear


In [ ]:
df['problem_category'].value_counts()

problem_category
unclear               2169
direction_confused     291
overwhelmed            242
beginner_lost          130
confidence_issue       110
Name: count, dtype: int64

In [ ]:
df['current_level'].value_counts()

current_level
unclear         2405
intermediate     242
basic            209
zero              86
Name: count, dtype: int64

In [ ]:
df['blocker_type'].value_counts()

blocker_type
unclear             2731
no_time              106
no_confidence         69
no_foundation         32
no_portfolio           3
too_many_options       1
Name: count, dtype: int64

In [2]:
import pandas as pd
import numpy as np

from tqdm import tqdm

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


# =========================================================
# LOAD DATA
# =========================================================

df = pd.read_csv(
    "career_role_prediction.csv"
)

print("Data:", len(df))


# =========================================================
# CLEAN TEXT
# =========================================================

df["title"] = (
    df["title"]
    .fillna("")
    .astype(str)
    .str.lower()
)

df["body"] = (
    df["body"]
    .fillna("")
    .astype(str)
    .str.lower()
)

df["text"] = (
    df["title"] + " " +
    df["body"]
)


# =========================================================
# LOAD MODEL
# =========================================================

print("\nLoading model...")

model = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2"
)


# =========================================================
# REFERENCE EXAMPLES
# =========================================================

problem_examples = {

    "confidence_issue": [

        "i am not confident",
        "i doubt myself",
        "i feel insecure",
        "i compare myself to others",
        "i feel stupid",
        "i am scared to apply",
        "i have imposter syndrome"

    ],

    "overwhelmed": [

        "there is too much to learn",
        "i feel overwhelmed",
        "too many things to study",
        "i dont know what to learn first",
        "everything feels confusing",
        "my roadmap is messy"

    ],

    "direction_confused": [

        "i dont know which path to choose",
        "frontend or backend",
        "which tech career is best",
        "i keep changing career path",
        "i am confused about my future"

    ],

    "beginner_lost": [

        "where do i start",
        "i am a complete beginner",
        "i have zero experience",
        "i dont know how to start coding",
        "i am new to tech"

    ]

}


# =========================================================
# LEVEL EXAMPLES
# =========================================================

level_examples = {

    "zero": [

        "i have zero experience",
        "i never coded before",
        "complete beginner",
        "starting from zero"

    ],

    "basic": [

        "i finished tutorials",
        "i know basic python",
        "i made small projects",
        "i joined bootcamp"

    ],

    "intermediate": [

        "i have internship experience",
        "i worked on projects",
        "i am intermediate",
        "i freelance sometimes"

    ]

}


# =========================================================
# BLOCKER EXAMPLES
# =========================================================

blocker_examples = {

    "too_many_options": [

        "too many choices",
        "too many roadmaps",
        "i keep changing path",
        "i cannot decide",
        "i jump between topics"

    ],

    "no_portfolio": [

        "i have no projects",
        "my portfolio is empty",
        "nothing to show employers",
        "i lack practical projects"

    ],

    "no_time": [

        "i work full time",
        "i am busy every day",
        "i dont have time to study",
        "my schedule is packed"

    ],

    "no_confidence": [

        "i doubt myself",
        "i feel insecure",
        "i compare myself to others",
        "i am afraid to apply"

    ],

    "no_foundation": [

        "my fundamentals are weak",
        "i dont understand basics",
        "my coding basics are bad",
        "foundation is weak"

    ]

}


# =========================================================
# GENERIC FUNCTION
# =========================================================

def predict_label(text, label_examples):

    # =====================================================
    # ENCODE INPUT
    # =====================================================

    text_emb = model.encode(
        [text],
        normalize_embeddings=True
    )


    best_label = None
    best_score = -1


    # =====================================================
    # LOOP LABELS
    # =====================================================

    for label, examples in label_examples.items():

        example_embs = model.encode(
            examples,
            normalize_embeddings=True
        )

        sims = cosine_similarity(
            text_emb,
            example_embs
        )[0]

        score = np.mean(
            np.sort(sims)[-2:]
        )

        if score > best_score:

            best_score = score
            best_label = label


    return best_label, round(float(best_score), 4)


# =========================================================
# PREDICTION
# =========================================================

problem_labels = []
level_labels = []
blocker_labels = []

print("\nPredicting labels...")

for text in tqdm(df["text"]):

    # =====================================================
    # PROBLEM
    # =====================================================

    problem, _ = predict_label(
        text,
        problem_examples
    )


    # =====================================================
    # LEVEL
    # =====================================================

    level, _ = predict_label(
        text,
        level_examples
    )


    # =====================================================
    # BLOCKER
    # =====================================================

    blocker, _ = predict_label(
        text,
        blocker_examples
    )


    # =====================================================
    # SAVE
    # =====================================================

    problem_labels.append(problem)

    level_labels.append(level)

    blocker_labels.append(blocker)


# =========================================================
# SAVE RESULT
# =========================================================

df["problem_category"] = problem_labels

df["current_level"] = level_labels

df["blocker_type"] = blocker_labels


# =========================================================
# PREVIEW
# =========================================================

preview_cols = [

    "title",
    "predicted_role",
    "problem_category",
    "current_level",
    "blocker_type"

]

print("\nTOP RESULTS:\n")

print(
    df[preview_cols]
    .head(20)
)


# =========================================================
# SAVE CSV
# =========================================================

output_path = "career_final_dataset.csv"

df.to_csv(

    output_path,

    index=False,
    encoding="utf-8-sig"

)

print("\nDONE!")
print("Saved to:", output_path)


Data: 2942

Loading model...

Predicting labels...


100%|██████████| 2942/2942 [1:21:37<00:00,  1.66s/it]



TOP RESULTS:

                                                title  \
0        what are the skills required for vfx artist?   
1   web developers out there can you provide guida...   
2   interested in developing a career in the busin...   
3   what are possible career s paths for data busi...   
4   what are the best opportunities for a manageme...   
5     what should i learn for mobile app development?   
6   what are possible career paths for a data busi...   
7                     what is the demand for powerbi?   
8   please help me out to build my career as a web...   
9   data analysts what are the most important exce...   
10  how can data analytics skills be utilized for ...   
11  subject combination for someone going for soft...   
12  what do i need to learn to transition to a car...   
13      jobs for data driven and analytical thinkers?   
14  how can i learn what being a business intellig...   
15  what job would let me get data from a database...   
16    web develo